In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.5 MB/s eta 0:00:00


In [2]:
import anthropic
from getpass import getpass
import json

# Ключ вводится вручную, не хранится в коде — безопасно
api_key = getpass("Вставьте ваш ANTHROPIC_API_KEY: ")
client = anthropic.Anthropic(api_key=api_key)

MODEL_ID = "claude-opus-5"
MAX_TOKENS = 2048

CORPUS = {
    "sentence": {
        "en": "The bank raised interest rates by two percentage points last quarter.",
        "ru": "Банк повысил процентные ставки на два процентных пункта в прошлом квартале.",
        "kk": "Банк өткен тоқсанда пайыздық мөлшерлемені екі пайыздық тармаққа көтерді."
    },
    "complaint": {
        "en": "Good afternoon. I opened a deposit at your branch in March and was told the rate was fixed for twelve months. In August the rate on my account dropped without any notice. I have attached the contract and the statement. Please explain on what basis the rate was changed and restore the original terms.",
        "ru": "Добрый день. Я открыл депозит в вашем отделении в марте, и мне сказали, что ставка зафиксирована на двенадцать месяцев. В августе ставка по моему счёту снизилась без какого-либо уведомления. Прилагаю договор и выписку. Прошу объяснить, на каком основании была изменена ставка, и восстановить первоначальные условия.",
        "kk": "Қайырлы күн. Мен наурыз айында сіздің бөлімшеңізде депозит аштым, маған мөлшерлеме он екі айға бекітілген деп айтылды. Тамыз айында менің шотымдағы мөлшерлеме ешқандай хабарламасыз төмендеді. Шартты және үзінді көшірмені қоса тіркеп отырмын. Мөлшерлеме қандай негізде өзгертілгенін түсіндіріп, бастапқы шарттарды қалпына келтіруіңізді сұраймын."
    },
    "system_prompt": {
        "en": "You are a support assistant for a retail bank. Answer only from the documents provided. If the answer is not in them, say so. Never invent an account number, a rate or a date.",
        "ru": "Вы — ассистент поддержки розничного банка. Отвечайте только по предоставленным документам. Если ответа в них нет, так и скажите. Никогда не выдумывайте номер счёта, ставку или дату.",
        "kk": "Сіз — бөлшек банктің қолдау көрсету ассистентісіз. Тек берілген құжаттар бойынша жауап беріңіз. Егер жауап оларда болмаса, солай деп айтыңыз. Шот нөмірін, мөлшерлемені немесе күнді ешқашан ойдан шығармаңыз."
    }
}
LANGUAGES = ["en", "ru", "kk"]

def count_tokens(text):
    result = client.messages.count_tokens(model=MODEL_ID, messages=[{"role": "user", "content": text}])
    return result.input_tokens

def count_request_tokens(lang):
    result = client.messages.count_tokens(
        model=MODEL_ID,
        system=CORPUS["system_prompt"][lang],
        messages=[{"role": "user", "content": CORPUS["complaint"][lang]}],
    )
    return result.input_tokens

def one_real_request(lang):
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=MAX_TOKENS,
        system=CORPUS["system_prompt"][lang],
        messages=[{"role": "user", "content": CORPUS["complaint"][lang]}],
    )
    if response.stop_reason == "refusal":
        print(f"  model declined: {response.stop_details}")
        return None
    print(f"  stop_reason: {response.stop_reason}")
    for block in response.content:
        if block.type == "text":
            print("  --- answer ---")
            print("  " + block.text.replace("\n", "\n  "))
    usage = response.usage
    print(f"  billed: {usage.input_tokens} in, {usage.output_tokens} out")
    return {"input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens}

# --- Считаем токены по всему корпусу (бесплатно) ---
counts = {}
print(f"counting tokens on {MODEL_ID} (free, no model run)")
for item_id, versions in CORPUS.items():
    counts[item_id] = {lang: count_tokens(versions[lang]) for lang in LANGUAGES}
    row = " ".join(f"{lang}={counts[item_id][lang]}" for lang in LANGUAGES)
    print(f"  {item_id:<14} {row}")

request_tokens = {lang: count_request_tokens(lang) for lang in LANGUAGES}
row = " ".join(f"{lang}={request_tokens[lang]}" for lang in LANGUAGES)
print(f"  {'request':<14} {row}")

# --- Реальный запрос модели (платно, копейки) ---
billed = {}
print(f"\nanswering the same complaint on {MODEL_ID}, in each language:")
for lang in LANGUAGES:
    print(f"\n[{lang}]")
    result = one_real_request(lang)
    if result is not None:
        billed[lang] = result

payload = {
    "model": "opus-5",
    "model_id": MODEL_ID,
    "token_counts": counts,
    "request_tokens": request_tokens,
    "one_request_billed": billed or None,
}

with open("measurements.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nГотово! measurements.json сохранён с настоящими данными Claude.")

Вставьте ваш ANTHROPIC_API_KEY: ··········
counting tokens on claude-opus-5 (free, no model run)
  sentence       en=26 ru=35 kk=48
  complaint      en=92 ru=134 kk=198
  system_prompt  en=58 ru=80 kk=124
  request        en=145 ru=209 kk=317

answering the same complaint on claude-opus-5, in each language:

[en]
  stop_reason: end_turn
  --- answer ---
  Thank you for getting in touch, and I'm sorry for the trouble this has caused.
  
  **I need to flag one thing first:** although you mention attaching the contract and the statement, no documents have reached me in this conversation. I'm not able to see them, and I can't answer questions about your rate, your terms, or the reason for the change without them. I won't guess at what your contract says.
  
  **What I'd need to review your case:**
  
  - The deposit agreement signed in March, including any terms and conditions or schedules referenced in it
  - The account statement showing the rate before and after the change in August
  -